In [1]:
# Deploying LLMs with 03-llama.cpp
#
# Goal:
# Learn how to serve a GGUF model locally using Docker and CUDA.

In [2]:
# Verify Docker installation
!docker --version

Docker version 29.6.1, build 8900f1d


In [3]:
# Verify GPU access inside Docker
!docker run --rm --gpus all nvidia/cuda:12.8.0-runtime-ubuntu22.04 nvidia-smi


== CUDA ==

CUDA Version 12.8.0

Container image Copyright (c) 2016-2023, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This container image and its contents are governed by the NVIDIA Deep Learning Container License.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/ngc/nvidia-deep-learning-container-license

A copy of this license is made available in this container at /NGC-DL-CONTAINER-LICENSE for your convenience.

Sat Jul  4 15:46:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.102.01             Driver Version: 581.57         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|      

## Download a GGUF Model

Example:

TinyLlama 1.1B Chat

```bash
python -m huggingface_hub download TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF tinyllama-1.1b-chat-v1.0.Q2_K.gguf --local-dir C:\Users\Shubham\llama_models\models
```

## Launch llama.cpp

```bash
docker run --rm --gpus all -p 8080:8080 -v C:\Users\Shubham\llama_models\models:/models ghcr.io/ggml-org/llama.cpp:server-cuda -m /models/tinyllama-1.1b-chat-v1.0.Q2_K.gguf
```

## Access the Chat UI

Open:

```
http://localhost:8080
```

The built-in chat interface is served by llama.cpp.

In [5]:
import requests

requests.get("http://localhost:8080/v1/models").json()

ConnectionError: HTTPConnectionPool(host='localhost', port=8080): Max retries exceeded with url: /v1/models (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8080): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))

In [ ]:
import requests

response = requests.post(
    "http://localhost:8080/v1/chat/completions",
    json={
        "messages": [
            {
                "role": "user",
                "content": "Explain Docker in one sentence."
            }
        ]
    }
)

print(response.json())

## Expose Publicly

Install Cloudflare Tunnel

```bash
winget install Cloudflare.cloudflared
```

Create a public endpoint

```bash
cloudflared tunnel --url http://localhost:8080
```

Cloudflare returns a public HTTPS URL that securely forwards traffic to your local llama.cpp server.

Key Takeaways

✔ GGUF models
✔ llama.cpp
✔ Docker
✔ CUDA
✔ OpenAI-compatible APIs
✔ Cloudflare Tunnel
✔ Local LLM deployment